# T2 — Recuperação de Informação (SCC0282): lexical + semântico + híbrido (RRF) + reranking + RAG no NFCorpus

Notebook-guia: executa os scripts na ordem recomendada. Tudo roda no Colab (GPU opcional, só necessária para a LLM local do RAG) ou localmente.

**Ordem:** `00` dados → `01` ajustes no dev → `02` pipeline no test → `03` erros ortográficos → `04` análises por consulta → `05` RAG (contexto/prompts) → `06` RAG (LLM) → `07` avaliação humana → `08` tabelas/figuras do relatório.

## 0. Ambiente

In [ ]:
# Colab: clone/descompacte o projeto, entre na pasta e instale as dependências
# %cd T2_RI
!pip install -q -r requirements.txt
import nltk; nltk.download('stopwords', quiet=True)

## 1. Dados (NFCorpus/BEIR) — valida contagens e a lista das 10 consultas do RAG (seed 42)

In [ ]:
!python scripts/00_download_data.py

## 2. Ajustes no DEV (pré-processamento, BM25, ortografia, bi-encoder, k do RRF, cross-encoder)
Usa `configs/models_hf.json` (sentence-transformers). Sem PyTorch/HF: `RI_MODELS=configs/models_local.json` (ver README).

In [ ]:
!python scripts/01_dev_tuning.py --stage all

## 3. Pipeline no TEST: lexical, semântico, híbrido e híbrido + reranking (+ CSVs padronizados)

In [ ]:
!python scripts/02_run_test_pipeline.py

In [ ]:
import pandas as pd
pd.read_csv('results/summary_test.csv')

## 4. Tolerância a erros (10 consultas do test)

In [ ]:
!python scripts/03_typo_experiment.py

## 5. Análises por consulta (lexical×semântico; reranking) — gera as evidências para o relatório

In [ ]:
!python scripts/04_analysis_cases.py > /dev/null
print(open('results/analysis/lex_vs_sem_cases.md').read()[:3000])

## 6. RAG
### 6.1 Contexto Top-5 pós-reranking, métricas @5, prompts e rastreio de falhas

In [ ]:
!python scripts/05_rag_prepare.py

### 6.2 Geração (mesma LLM e mesmos parâmetros nas condições A e B)
Escolha **um** backend. Exemplos:
- Colab com GPU: `--backend hf --model Qwen/Qwen2.5-3B-Instruct`
- API da Anthropic: `--backend anthropic --model <modelo>` (variável `ANTHROPIC_API_KEY`)
- API compatível com OpenAI / Ollama / vLLM: `--backend openai --model <modelo> [--base-url ...]`
- Respostas já geradas (JSONL): `--backend manual --answers-file results/rag_answers_manual.jsonl`

In [ ]:
!python scripts/06_rag_generate.py --backend hf --model Qwen/Qwen2.5-3B-Instruct

### 6.3 Avaliação humana (0/1/2) — planilha cega e embaralhada
1. `make-sheet` cria `results/rag_annotation_sheet.csv` (sem a condição A/B).
2. Cada avaliador preenche uma cópia (idealmente dois, de forma independente).
3. `merge` grava as notas em `results/rag_experiment.csv` e calcula médias e concordância.

In [ ]:
!python scripts/07_rag_evaluate.py make-sheet

In [ ]:
# depois de preencher as planilhas:
# !python scripts/07_rag_evaluate.py merge --sheets avaliador1.csv avaliador2.csv

## 7. Testes e tabelas/figuras do relatório

In [ ]:
!python -m pytest -q tests

In [ ]:
!python scripts/08_make_report_assets.py